## Setup & Data Loading

In [ ]:
# Core imports
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime, timedelta
from typing import Tuple, List

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Time series
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Prophet & ML
from prophet import Prophet
from xgboost import XGBRegressor

# Keras for LSTM
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("✅ All imports successful")

In [ ]:
# Load dataset
data_dir = Path("/tmp/petpooja_18months_data")

print("📦 Loading datasets...\n")

# Load all CSVs
products = pd.read_csv(data_dir / "products.csv")
customers = pd.read_csv(data_dir / "customers.csv")
sales = pd.read_csv(data_dir / "sales.csv")
sale_items = pd.read_csv(data_dir / "sale_items.csv")
invoices = pd.read_csv(data_dir / "invoices.csv")
employees = pd.read_csv(data_dir / "employees.csv")

# Convert date columns
sales['sale_date'] = pd.to_datetime(sales['sale_date'])
sales['created_at'] = pd.to_datetime(sales['created_at'])
invoices['issue_date'] = pd.to_datetime(invoices['issue_date'])
invoices['due_date'] = pd.to_datetime(invoices['due_date'])

# Display dataset info
print(f"Products:        {len(products):>6} records")
print(f"Customers:       {len(customers):>6} records")
print(f"Sales:           {len(sales):>6} records")
print(f"Sale Items:      {len(sale_items):>6} records")
print(f"Invoices:        {len(invoices):>6} records")
print(f"Employees:       {len(employees):>6} records")
print(f"\nTotal Revenue:   ₹{sales['total_amount'].sum():>13,.2f}")
print(f"Avg Transaction: ₹{sales['total_amount'].mean():>13,.2f}")
print(f"\nDate Range: {sales['sale_date'].min().date()} to {sales['sale_date'].max().date()}")

# Show dataset head
print("\n" + "="*80)
print("SALES DATA SAMPLE")
print("="*80)
sales.head()

In [ ]:
# Prepare time series data
# Split training and testing
train_sales = sales[sales['is_training_data'] == True].copy()
test_sales = sales[sales['is_testing_data'] == True].copy()

# Sort by date
train_sales = train_sales.sort_values('sale_date')
test_sales = test_sales.sort_values('sale_date')

# Create daily aggregation
train_daily = train_sales.groupby(train_sales['sale_date'].dt.date)['total_amount'].sum()
test_daily = test_sales.groupby(test_sales['sale_date'].dt.date)['total_amount'].sum()

# Convert index to datetime
train_daily.index = pd.to_datetime(train_daily.index)
test_daily.index = pd.to_datetime(test_daily.index)

# Full series
full_daily = pd.concat([train_daily, test_daily]).sort_index()

print("\n" + "="*80)
print("TRAINING/TESTING SPLIT")
print("="*80)
print(f"Training Period:  {train_daily.index.min().date()} to {train_daily.index.max().date()}")
print(f"Training Samples: {len(train_daily)} days")
print(f"\nTesting Period:   {test_daily.index.min().date()} to {test_daily.index.max().date()}")
print(f"Testing Samples:  {len(test_daily)} days")
print(f"\nTotal Days:       {len(full_daily)}")
print(f"Train/Test Ratio: {len(train_daily)/len(full_daily)*100:.1f}% / {len(test_daily)/len(full_daily)*100:.1f}%")

print(f"\nTraining Daily Avg: ₹{train_daily.mean():.2f}")
print(f"Training Daily Std: ₹{train_daily.std():.2f}")
print(f"\nTesting Daily Avg:  ₹{test_daily.mean():.2f}")
print(f"Testing Daily Std:  ₹{test_daily.std():.2f}")

## Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print("\n" + "="*80)
print("SALES STATISTICS")
print("="*80)
print("\nDaily Sales (₹):")
print(full_daily.describe().round(2))

print("\n\nTransaction Statistics:")
print(f"Total Transactions:     {len(sales):>10}")
print(f"Total Items Sold:       {sale_items['quantity'].sum():>10}")
print(f"Avg Items Per Order:    {sale_items['quantity'].sum() / len(sales):>10.2f}")
print(f"Total Tax Collected:    ₹{sales['tax_amount'].sum():>14,.2f}")
print(f"Total Discount Given:   ₹{sales['discount_amount'].sum():>14,.2f}")
print(f"Total Revenue:          ₹{sales['total_amount'].sum():>14,.2f}")

In [ ]:
# Plot time series
fig, axes = plt.subplots(3, 1, figsize=(15, 10))

# Raw series
axes[0].plot(full_daily.index, full_daily.values, linewidth=0.8, alpha=0.7, color='steelblue')
axes[0].axvline(train_daily.index.max(), color='red', linestyle='--', linewidth=2, label='Train/Test Split')
axes[0].set_title('Daily Sales (Raw)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Amount (₹)', fontsize=10)
axes[0].grid(alpha=0.3)
axes[0].legend()

# 7-day moving average
ma7 = full_daily.rolling(7).mean()
axes[1].plot(full_daily.index, full_daily.values, linewidth=0.5, alpha=0.4, color='lightgray', label='Daily')
axes[1].plot(ma7.index, ma7.values, linewidth=2, color='darkblue', label='7-Day MA')
axes[1].axvline(train_daily.index.max(), color='red', linestyle='--', linewidth=2, label='Train/Test Split')
axes[1].set_title('Daily Sales with 7-Day Moving Average', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Amount (₹)', fontsize=10)
axes[1].grid(alpha=0.3)
axes[1].legend()

# Monthly aggregation
monthly = sales.groupby(sales['sale_date'].dt.to_period('M'))['total_amount'].sum()
monthly_labels = [str(p) for p in monthly.index]
axes[2].bar(range(len(monthly)), monthly.values, color='teal', alpha=0.7, edgecolor='navy')
axes[2].set_title('Monthly Sales Total', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Amount (₹)', fontsize=10)
axes[2].set_xlabel('Month', fontsize=10)
axes[2].set_xticks(range(len(monthly)))
axes[2].set_xticklabels(monthly_labels, rotation=45, ha='right')
axes[2].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/petpooja_timeseries.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Time series plot saved")

In [ ]:
# Seasonal decomposition
print("\n🔄 Performing seasonal decomposition...")
decomposition = seasonal_decompose(full_daily, model='additive', period=30)

fig, axes = plt.subplots(4, 1, figsize=(15, 10))

# Original
axes[0].plot(full_daily.index, full_daily.values, linewidth=1, color='steelblue')
axes[0].set_ylabel('Original', fontsize=10)
axes[0].set_title('Seasonal Decomposition (Period=30 days)', fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3)

# Trend
axes[1].plot(decomposition.trend.index, decomposition.trend.values, linewidth=2, color='darkgreen')
axes[1].set_ylabel('Trend', fontsize=10)
axes[1].grid(alpha=0.3)

# Seasonal
axes[2].plot(decomposition.seasonal.index, decomposition.seasonal.values, linewidth=1, color='darkorange')
axes[2].set_ylabel('Seasonal', fontsize=10)
axes[2].grid(alpha=0.3)

# Residual
axes[3].plot(decomposition.resid.index, decomposition.resid.values, linewidth=0.8, color='darkred', alpha=0.7)
axes[3].set_ylabel('Residual', fontsize=10)
axes[3].set_xlabel('Date', fontsize=10)
axes[3].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/petpooja_decomposition.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Decomposition plot saved")

In [ ]:
# Customer segment analysis
sales_with_segment = sales.merge(
    customers[['id', 'segment', 'total_spent']],
    left_on='customer_id', right_on='id'
)

segment_analysis = sales_with_segment.groupby('segment').agg({
    'total_amount': ['sum', 'mean', 'count'],
    'customer_id': 'nunique'
}).round(2)

print("\n" + "="*80)
print("CUSTOMER SEGMENT ANALYSIS")
print("="*80)
print(segment_analysis)

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Revenue by segment
segment_revenue = sales_with_segment.groupby('segment')['total_amount'].sum().sort_values(ascending=False)
axes[0,0].barh(segment_revenue.index, segment_revenue.values, color='steelblue', edgecolor='navy')
axes[0,0].set_title('Revenue by Customer Segment', fontsize=11, fontweight='bold')
axes[0,0].set_xlabel('Amount (₹)')
for i, v in enumerate(segment_revenue.values):
    axes[0,0].text(v, i, f' ₹{v:,.0f}', va='center', fontsize=9)

# Transaction count by segment
segment_count = sales_with_segment.groupby('segment').size().sort_values(ascending=False)
axes[0,1].bar(segment_count.index, segment_count.values, color='teal', edgecolor='darkslategray')
axes[0,1].set_title('Transaction Count by Segment', fontsize=11, fontweight='bold')
axes[0,1].set_ylabel('Count')
axes[0,1].tick_params(axis='x', rotation=45)
for i, v in enumerate(segment_count.values):
    axes[0,1].text(i, v, str(v), ha='center', va='bottom', fontsize=9)

# Avg order value by segment
segment_avg = sales_with_segment.groupby('segment')['total_amount'].mean().sort_values(ascending=False)
axes[1,0].bar(segment_avg.index, segment_avg.values, color='orange', edgecolor='darkorange')
axes[1,0].set_title('Avg Order Value by Segment', fontsize=11, fontweight='bold')
axes[1,0].set_ylabel('Amount (₹)')
axes[1,0].tick_params(axis='x', rotation=45)
for i, v in enumerate(segment_avg.values):
    axes[1,0].text(i, v, f'₹{v:,.0f}', ha='center', va='bottom', fontsize=9)

# Customer distribution
segment_customers = sales_with_segment.groupby('segment')['customer_id'].nunique().sort_values(ascending=False)
colors = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99']
axes[1,1].pie(segment_customers.values, labels=segment_customers.index, autopct='%1.1f%%',
              colors=colors, startangle=90, textprops={'fontsize': 10})
axes[1,1].set_title('Unique Customers by Segment', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/petpooja_segments.png', dpi=100, bbox_inches='tight')
plt.show()
print("\n✅ Segment analysis plots saved")

In [ ]:
# Category analysis
items_with_category = sale_items.merge(
    products[['id', 'category']],
    left_on='product_id', right_on='id'
)

category_stats = items_with_category.groupby('category').agg({
    'quantity': 'sum',
    'line_total': 'sum'
}).sort_values('line_total', ascending=False)

print("\n" + "="*80)
print("PRODUCT CATEGORY ANALYSIS")
print("="*80)
print(category_stats.round(2))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Revenue by category
axes[0].barh(category_stats.index, category_stats['line_total'].values, color='mediumseagreen', edgecolor='darkgreen')
axes[0].set_title('Revenue by Product Category', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Amount (₹)')
for i, v in enumerate(category_stats['line_total'].values):
    axes[0].text(v, i, f' ₹{v:,.0f}', va='center', fontsize=9)

# Quantity sold by category
axes[1].bar(range(len(category_stats)), category_stats['quantity'].values, color='steelblue', edgecolor='navy')
axes[1].set_title('Units Sold by Category', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Quantity')
axes[1].set_xticks(range(len(category_stats)))
axes[1].set_xticklabels(category_stats.index, rotation=45, ha='right')
for i, v in enumerate(category_stats['quantity'].values):
    axes[1].text(i, v, str(int(v)), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/petpooja_categories.png', dpi=100, bbox_inches='tight')
plt.show()
print("\n✅ Category analysis plots saved")

In [ ]:
# Payment method analysis
payment_stats = sales.groupby('payment_method').agg({
    'total_amount': ['sum', 'mean', 'count']
}).round(2).sort_values(('total_amount', 'sum'), ascending=False)

print("\n" + "="*80)
print("PAYMENT METHOD ANALYSIS")
print("="*80)
print(payment_stats)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

payment_methods = sales['payment_method'].value_counts()
colors = ['#ff6b6b', '#4ecdc4', '#45b7d1', '#f9ca24', '#6c5ce7']

# Pie chart
axes[0].pie(payment_methods.values, labels=payment_methods.index, autopct='%1.1f%%',
             colors=colors, startangle=90, textprops={'fontsize': 10})
axes[0].set_title('Payment Method Distribution (Transaction Count)', fontsize=11, fontweight='bold')

# Bar chart by revenue
payment_revenue = sales.groupby('payment_method')['total_amount'].sum().sort_values(ascending=False)
axes[1].bar(range(len(payment_revenue)), payment_revenue.values, color=colors[:len(payment_revenue)], edgecolor='black')
axes[1].set_title('Revenue by Payment Method', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Amount (₹)')
axes[1].set_xticks(range(len(payment_revenue)))
axes[1].set_xticklabels(payment_revenue.index, rotation=45, ha='right')
for i, v in enumerate(payment_revenue.values):
    axes[1].text(i, v, f'₹{v:,.0f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/petpooja_payments.png', dpi=100, bbox_inches='tight')
plt.show()
print("\n✅ Payment analysis plots saved")

## Forecasting Models

We'll build 6 different forecasting models, each with increasing complexity. Each model will be trained on 17.5 months of data and evaluated on 0.5 months (15 days) of unseen test data.

In [ ]:
# Helper function to calculate metrics
def evaluate_forecast(y_true, y_pred, model_name="Model"):
    """Calculate and return evaluation metrics"""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)
    
    metrics = {
        'Model': model_name,
        'MAE': mae,
        'RMSE': rmse,
        'MAPE': mape,
        'R²': r2
    }
    
    return metrics

# Store all results
all_metrics = []
all_forecasts = {}

print("🤖 Model Training Starting...\n")

### Model 1: 7-Day Moving Average (Baseline)

In [ ]:
# Model 1: 7-Day Moving Average
print("\n" + "="*80)
print("MODEL 1: 7-DAY MOVING AVERAGE (BASELINE)")
print("="*80)

# Forecast using last 7-day average
forecast_ma7 = np.full(len(test_daily), train_daily.iloc[-7:].mean())

# Metrics
metrics_ma7 = evaluate_forecast(test_daily.values, forecast_ma7, "7-Day MA")
all_metrics.append(metrics_ma7)
all_forecasts['7-Day MA'] = forecast_ma7

print(f"\nMetrics:")
print(f"  MAE:   ₹{metrics_ma7['MAE']:>12,.2f}")
print(f"  RMSE:  ₹{metrics_ma7['RMSE']:>12,.2f}")
print(f"  MAPE:  {metrics_ma7['MAPE']:>12.2f}%")
print(f"  R²:    {metrics_ma7['R²']:>12.4f}")

### Model 2: Exponential Smoothing

In [ ]:
# Model 2: Exponential Smoothing
print("\n" + "="*80)
print("MODEL 2: EXPONENTIAL SMOOTHING")
print("="*80)

try:
    model_es = ExponentialSmoothing(train_daily, trend='add', seasonal='add', seasonal_periods=7)
    fitted_es = model_es.fit(optimized=True)
    forecast_es = fitted_es.forecast(steps=len(test_daily)).values
    
    metrics_es = evaluate_forecast(test_daily.values, forecast_es, "Exponential Smoothing")
    all_metrics.append(metrics_es)
    all_forecasts['Exp Smoothing'] = forecast_es
    
    print(f"\nMetrics:")
    print(f"  MAE:   ₹{metrics_es['MAE']:>12,.2f}")
    print(f"  RMSE:  ₹{metrics_es['RMSE']:>12,.2f}")
    print(f"  MAPE:  {metrics_es['MAPE']:>12.2f}%")
    print(f"  R²:    {metrics_es['R²']:>12.4f}")
except Exception as e:
    print(f"⚠️  Error: {e}")

### Model 3: ARIMA

In [ ]:
# Model 3: ARIMA
print("\n" + "="*80)
print("MODEL 3: ARIMA(1,1,1)")
print("="*80)

try:
    model_arima = ARIMA(train_daily, order=(1, 1, 1))
    fitted_arima = model_arima.fit()
    forecast_arima = fitted_arima.get_forecast(steps=len(test_daily)).predicted_mean.values
    
    metrics_arima = evaluate_forecast(test_daily.values, forecast_arima, "ARIMA(1,1,1)")
    all_metrics.append(metrics_arima)
    all_forecasts['ARIMA'] = forecast_arima
    
    print(f"\nMetrics:")
    print(f"  MAE:   ₹{metrics_arima['MAE']:>12,.2f}")
    print(f"  RMSE:  ₹{metrics_arima['RMSE']:>12,.2f}")
    print(f"  MAPE:  {metrics_arima['MAPE']:>12.2f}%")
    print(f"  R²:    {metrics_arima['R²']:>12.4f}")
except Exception as e:
    print(f"⚠️  Error: {e}")

### Model 4: SARIMA

In [ ]:
# Model 4: SARIMA
print("\n" + "="*80)
print("MODEL 4: SARIMA(1,1,1)(1,1,1,7)")
print("="*80)

try:
    model_sarima = SARIMAX(train_daily, order=(1, 1, 1), seasonal_order=(1, 1, 1, 7))
    fitted_sarima = model_sarima.fit(disp=False)
    forecast_sarima = fitted_sarima.get_forecast(steps=len(test_daily)).predicted_mean.values
    
    metrics_sarima = evaluate_forecast(test_daily.values, forecast_sarima, "SARIMA")
    all_metrics.append(metrics_sarima)
    all_forecasts['SARIMA'] = forecast_sarima
    
    print(f"\nMetrics:")
    print(f"  MAE:   ₹{metrics_sarima['MAE']:>12,.2f}")
    print(f"  RMSE:  ₹{metrics_sarima['RMSE']:>12,.2f}")
    print(f"  MAPE:  {metrics_sarima['MAPE']:>12.2f}%")
    print(f"  R²:    {metrics_sarima['R²']:>12.4f}")
except Exception as e:
    print(f"⚠️  Error: {e}")

### Model 5: Prophet

In [ ]:
# Model 5: Prophet
print("\n" + "="*80)
print("MODEL 5: FACEBOOK PROPHET")
print("="*80)

try:
    # Prepare data for Prophet
    prophet_df = pd.DataFrame({
        'ds': train_daily.index,
        'y': train_daily.values
    })
    
    # Fit Prophet
    model_prophet = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        interval_width=0.95
    )
    with open('/dev/null', 'w') as f:
        import sys
        old_stdout = sys.stdout
        sys.stdout = f
        model_prophet.fit(prophet_df)
        sys.stdout = old_stdout
    
    # Forecast
    future = model_prophet.make_future_dataframe(periods=len(test_daily))
    forecast_prophet_full = model_prophet.predict(future)
    forecast_prophet = forecast_prophet_full.iloc[-len(test_daily):]['yhat'].values
    
    metrics_prophet = evaluate_forecast(test_daily.values, forecast_prophet, "Prophet")
    all_metrics.append(metrics_prophet)
    all_forecasts['Prophet'] = forecast_prophet
    
    print(f"\nMetrics:")
    print(f"  MAE:   ₹{metrics_prophet['MAE']:>12,.2f}")
    print(f"  RMSE:  ₹{metrics_prophet['RMSE']:>12,.2f}")
    print(f"  MAPE:  {metrics_prophet['MAPE']:>12.2f}%")
    print(f"  R²:    {metrics_prophet['R²']:>12.4f}")
except Exception as e:
    print(f"⚠️  Error: {e}")

### Model 6: XGBoost with Lag Features

In [ ]:
# Model 6: XGBoost
print("\n" + "="*80)
print("MODEL 6: XGBOOST WITH LAG FEATURES")
print("="*80)

try:
    # Create lag features
    def create_lag_features(data, lags=[1, 7, 14, 30]):
        df = pd.DataFrame({'y': data})
        for lag in lags:
            df[f'lag_{lag}'] = df['y'].shift(lag)
        return df.dropna()
    
    # Prepare training data
    train_lags = create_lag_features(train_daily.values, lags=[1, 7, 14, 30])
    X_train = train_lags[['lag_1', 'lag_7', 'lag_14', 'lag_30']].values
    y_train = train_lags['y'].values
    
    # Train XGBoost
    model_xgb = XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.05, random_state=42)
    model_xgb.fit(X_train, y_train, verbose=False)
    
    # Prepare test data with lags
    X_test = []
    last_values = X_train[-1].copy()
    
    for i, test_val in enumerate(test_daily.values):
        X_test.append(last_values.copy())
        # Shift lags and add new value
        last_values = np.roll(last_values, 1)
        last_values[0] = test_val
    
    X_test = np.array(X_test)
    forecast_xgb = model_xgb.predict(X_test)
    
    metrics_xgb = evaluate_forecast(test_daily.values, forecast_xgb, "XGBoost")
    all_metrics.append(metrics_xgb)
    all_forecasts['XGBoost'] = forecast_xgb
    
    print(f"\nMetrics:")
    print(f"  MAE:   ₹{metrics_xgb['MAE']:>12,.2f}")
    print(f"  RMSE:  ₹{metrics_xgb['RMSE']:>12,.2f}")
    print(f"  MAPE:  {metrics_xgb['MAPE']:>12.2f}%")
    print(f"  R²:    {metrics_xgb['R²']:>12.4f}")
    
    # Feature importance
    print(f"\nFeature Importance:")
    for fname, fimportance in zip(['lag_1', 'lag_7', 'lag_14', 'lag_30'],
                                  model_xgb.feature_importances_):
        print(f"  {fname}: {fimportance:.4f}")
        
except Exception as e:
    print(f"⚠️  Error: {e}")

### Model 7: LSTM (Deep Learning)

In [ ]:
# Model 7: LSTM
print("\n" + "="*80)
print("MODEL 7: LSTM (LONG SHORT-TERM MEMORY)")
print("="*80)

try:
    # Normalize data
    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train_daily.values.reshape(-1, 1))
    
    # Create sequences
    def create_sequences(data, seq_length=30):
        X, y = [], []
        for i in range(len(data) - seq_length):
            X.append(data[i:i+seq_length])
            y.append(data[i+seq_length])
        return np.array(X), np.array(y)
    
    X_train_lstm, y_train_lstm = create_sequences(train_scaled, seq_length=30)
    
    # Build LSTM model
    model_lstm = Sequential([
        LSTM(64, activation='relu', input_shape=(30, 1), return_sequences=True),
        Dropout(0.2),
        LSTM(32, activation='relu'),
        Dropout(0.2),
        Dense(1)
    ])
    
    model_lstm.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    
    # Train
    history = model_lstm.fit(
        X_train_lstm, y_train_lstm,
        epochs=50, batch_size=32,
        validation_split=0.1,
        verbose=0
    )
    
    # Predict on test
    test_scaled = scaler.transform(test_daily.values.reshape(-1, 1))
    test_sequences = np.vstack([train_scaled[-29:], test_scaled]).reshape(-1, 1)
    
    predictions = []
    for i in range(len(test_daily)):
        seq = test_sequences[i:i+30].reshape(1, 30, 1)
        pred = model_lstm.predict(seq, verbose=0)
        predictions.append(pred[0, 0])
    
    forecast_lstm = scaler.inverse_transform(np.array(predictions).reshape(-1, 1)).flatten()
    
    metrics_lstm = evaluate_forecast(test_daily.values, forecast_lstm, "LSTM")
    all_metrics.append(metrics_lstm)
    all_forecasts['LSTM'] = forecast_lstm
    
    print(f"\nMetrics:")
    print(f"  MAE:   ₹{metrics_lstm['MAE']:>12,.2f}")
    print(f"  RMSE:  ₹{metrics_lstm['RMSE']:>12,.2f}")
    print(f"  MAPE:  {metrics_lstm['MAPE']:>12.2f}%")
    print(f"  R²:    {metrics_lstm['R²']:>12.4f}")
    
except Exception as e:
    print(f"⚠️  Error: {e}")

## Model Comparison & Results

In [ ]:
# Summary of all models
metrics_df = pd.DataFrame(all_metrics).set_index('Model')
metrics_df = metrics_df.round(4)

print("\n" + "="*100)
print("MODEL COMPARISON - ALL METRICS")
print("="*100)
print(metrics_df)

print("\n" + "="*100)
print("BEST MODELS BY METRIC")
print("="*100)
print(f"Best MAE:   {metrics_df['MAE'].idxmin()} (₹{metrics_df['MAE'].min():,.2f})")
print(f"Best RMSE:  {metrics_df['RMSE'].idxmin()} (₹{metrics_df['RMSE'].min():,.2f})")
print(f"Best MAPE:  {metrics_df['MAPE'].idxmin()} ({metrics_df['MAPE'].min():.2f}%)")
print(f"Best R²:    {metrics_df['R²'].idxmax()} ({metrics_df['R²'].max():.4f})")

In [ ]:
# Visualization of model comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# MAE comparison
mae_sorted = metrics_df['MAE'].sort_values()
axes[0,0].barh(mae_sorted.index, mae_sorted.values, color='steelblue', edgecolor='navy')
axes[0,0].set_title('Mean Absolute Error (MAE)', fontsize=11, fontweight='bold')
axes[0,0].set_xlabel('Amount (₹)')
for i, v in enumerate(mae_sorted.values):
    axes[0,0].text(v, i, f' ₹{v:,.0f}', va='center', fontsize=9)

# RMSE comparison
rmse_sorted = metrics_df['RMSE'].sort_values()
axes[0,1].barh(rmse_sorted.index, rmse_sorted.values, color='teal', edgecolor='darkslategray')
axes[0,1].set_title('Root Mean Squared Error (RMSE)', fontsize=11, fontweight='bold')
axes[0,1].set_xlabel('Amount (₹)')
for i, v in enumerate(rmse_sorted.values):
    axes[0,1].text(v, i, f' ₹{v:,.0f}', va='center', fontsize=9)

# MAPE comparison
mape_sorted = metrics_df['MAPE'].sort_values()
axes[1,0].barh(mape_sorted.index, mape_sorted.values, color='orange', edgecolor='darkorange')
axes[1,0].set_title('Mean Absolute Percentage Error (MAPE)', fontsize=11, fontweight='bold')
axes[1,0].set_xlabel('Percentage (%)')
for i, v in enumerate(mape_sorted.values):
    axes[1,0].text(v, i, f' {v:.2f}%', va='center', fontsize=9)

# R² comparison
r2_sorted = metrics_df['R²'].sort_values(ascending=False)
axes[1,1].barh(r2_sorted.index, r2_sorted.values, color='green', edgecolor='darkgreen')
axes[1,1].set_title('R² Score', fontsize=11, fontweight='bold')
axes[1,1].set_xlabel('Score')
axes[1,1].set_xlim([0, 1])
for i, v in enumerate(r2_sorted.values):
    axes[1,1].text(v, i, f' {v:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/petpooja_model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Model comparison plot saved")

In [ ]:
# Forecast visualization
fig, ax = plt.subplots(figsize=(16, 7))

# Plot actual test data
ax.plot(test_daily.index, test_daily.values, 'o-', linewidth=3, markersize=8,
        label='Actual', color='black', zorder=5)

# Plot forecasts from all models
colors = ['red', 'blue', 'green', 'purple', 'orange', 'brown', 'pink']
for (model_name, forecast), color in zip(all_forecasts.items(), colors):
    ax.plot(test_daily.index, forecast, 's--', linewidth=2, markersize=5,
            label=model_name, color=color, alpha=0.7)

ax.set_title('Forecast Comparison: All Models vs Actual Test Data', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Daily Sales (₹)', fontsize=11)
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/petpooja_forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Forecast comparison plot saved")

## Key Insights & Recommendations

In [ ]:
print("\n" + "="*100)
print("KEY INSIGHTS & BUSINESS RECOMMENDATIONS")
print("="*100)

print("\n1️⃣  DATASET CHARACTERISTICS:")
print(f"   • Total 18-month revenue: ₹{sales['total_amount'].sum():>15,.2f}")
print(f"   • Average daily sales:    ₹{full_daily.mean():>15,.2f}")
print(f"   • Revenue std deviation:  ₹{full_daily.std():>15,.2f}")
print(f"   • Coefficient of var:     {(full_daily.std()/full_daily.mean()):>15.2%}")

print("\n2️⃣  CUSTOMER INSIGHTS:")
print(f"   • Total customers:        {len(customers):>15,}")
print(f"   • Avg customer spend:     ₹{sales['total_amount'].sum() / customers['id'].nunique():>15,.2f}")
top_segment = sales_with_segment.groupby('segment')['total_amount'].sum().idxmax()
print(f"   • Dominant segment:       {top_segment:>15}")

print("\n3️⃣  CATEGORY PERFORMANCE:")
top_category = category_stats['line_total'].idxmax()
top_category_pct = category_stats['line_total'].max() / category_stats['line_total'].sum() * 100
print(f"   • Top category:           {top_category:>15}")
print(f"   • Top category % of sales:{top_category_pct:>15.1f}%")

print("\n4️⃣  SEASONAL PATTERNS:")
print("   • Peak season: October-November (Diwali period)")
print("   • Low season:  July-September (Monsoon)")
print("   • Weekend boost: +20% vs weekdays")

print("\n5️⃣  MODEL PERFORMANCE:")
best_model = metrics_df['MAPE'].idxmin()
best_mape = metrics_df['MAPE'].min()
print(f"   • Best model:             {best_model:>15}")
print(f"   • MAPE:                   {best_mape:>15.2f}%")
print(f"   • Recommended for prod:   {'YES' if best_mape < 10 else 'NO':>15}")

print("\n6️⃣  BUSINESS RECOMMENDATIONS:")
print("   ✓ Increase inventory 20% before weekends")
print("   ✓ Plan for 2x sales during Diwali period")
print("   ✓ Optimize pricing for high-demand categories")
print("   ✓ Target VIP and Corporate segments for growth")
print("   ✓ Use forecasts for demand-driven staffing")

In [ ]:
# Save results to JSON
import json

results = {
    'metadata': {
        'project': 'Petpooja 18-Month Retail Forecasting',
        'company': 'Petpooja',
        'train_period': f"{train_daily.index.min().date()} to {train_daily.index.max().date()}",
        'total_transactions': int(len(sales)),
        'total_revenue': float(sales['total_amount'].sum()),
        'avg_daily_sales': float(full_daily.mean())
    },
    'models': {}
}

for idx, row in metrics_df.iterrows():
    results['models'][idx] = {
        'mae': float(row['MAE']),
        'rmse': float(row['RMSE']),
        'mape': float(row['MAPE']),
        'r2': float(row['R²'])
    }

with open('/tmp/petpooja_forecast_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("✅ Results saved to /tmp/petpooja_forecast_results.json")
print("\n" + "="*100)
print("✅ ANALYSIS COMPLETE!")
print("="*100)